# Telegram plant navigation demo

This shows a location-first picker and the native command menu using the public fixture in `data/demo_garden.json`. Previously `/plants` presented one long row of plant buttons; now a large location is paged while an unassigned cutting remains findable. The menu names the existing controls, and `/start` offers immediate navigation buttons. Plant records and watering plans are unchanged; this demo uses mocks instead of a bot token or private household data.


In [1]:
import json
from pathlib import Path
from unittest.mock import AsyncMock, Mock
from uuid import uuid4

from her_garden.bot import BotSettings, GardenBot

fixture = json.loads(Path("data/demo_garden.json").read_text())
first_id, second_id = str(uuid4()), str(uuid4())
locations = [
    {"id": first_id, "name": fixture["location"]},
    {"id": second_id, "name": fixture["second_location"]},
]
plants = [
    {"id": str(uuid4()), "name": fixture["plant"]["name"], "location_id": first_id},
    *[
        {"id": str(uuid4()), "name": f"Crassula cutting {number:02}", "location_id": first_id}
        for number in range(1, 13)
    ],
    {"id": str(uuid4()), "name": fixture["second_plant"]["name"], "location_id": second_id},
    {"id": str(uuid4()), "name": "Monstera cutting"},
    {"id": str(uuid4()), "name": "Retired cutting", "status": "dead"},
]
bot = GardenBot(BotSettings("demo-only", frozenset(), "postgresql://unused"))
bot.garden.list_entities = AsyncMock(
    side_effect=lambda kind: plants if kind == "plant" else locations
)
update = Mock()
update.callback_query = Mock()
update.callback_query.edit_message_text = AsyncMock()

await bot._show_locations(update)
markup = update.callback_query.edit_message_text.call_args.kwargs["reply_markup"]
print("Location buttons:", [row[0].text for row in markup.inline_keyboard])
await bot._show_location(update, first_id, 0)
markup = update.callback_query.edit_message_text.call_args.kwargs["reply_markup"]
print("First page plant buttons:", len(markup.inline_keyboard) - 2)
await bot._show_location(update, first_id, 1)
markup = update.callback_query.edit_message_text.call_args.kwargs["reply_markup"]
print("Second page plant buttons:", len(markup.inline_keyboard) - 2)
await bot._show_location(update, "none", 0)
markup = update.callback_query.edit_message_text.call_args.kwargs["reply_markup"]
print("Without location:", markup.inline_keyboard[0][0].text)

app = Mock()
app.bot.set_my_commands = AsyncMock(return_value=True)
app.bot.set_chat_menu_button = AsyncMock(return_value=True)
await bot._configure_menu(app)
print("Telegram menu:", [command.command for command in app.bot.set_my_commands.call_args.args[0]])
print("Start buttons:", [row[0].text for row in bot._menu_keyboard().inline_keyboard])


Location buttons: ['bedroom shelf (1)', 'north balcony (13)', 'Без места (1)']
First page plant buttons: 12
Second page plant buttons: 1
Without location: Monstera cutting
Telegram menu: ['plants', 'time', 'help', 'cancel']
Start buttons: ['Растения', 'Общее время']
